In [ ]:
import os, shutil, pathlib, random
import numpy as np
import cv2

In [7]:
data_root = pathlib.Path("D:/FYP/dental-vision/ML/data/dentaldataset01/YOLO")

In [21]:
CLASS_NAMES = {
    0:  "Caries",
    6:  "Missing_Teeth",
    7:  "Periapical_Lesion",
    11: "Impacted_Tooth",
    13: "Bone_Loss",
}

In [ ]:
OUTPUT = pathlib.Path("D:/FYP/dental-vision/ML/data/processed_cropped")

In [ ]:
def move_bone_loss_to_test():
    def get_files_with_class(split_name, target_class):
        label_dir = data_root / split_name / "labels"
        matching_files = []
        for lf in sorted(label_dir.glob("*.txt")):
            with open(lf, "r") as handle:
                for line in handle:
                    parts = line.strip().split()
                    if parts and int(parts[0]) == target_class:
                        matching_files.append(lf)
                        break
        return matching_files

    train_bone_loss_labels = get_files_with_class("train", 13)
    test_bone_loss_labels = get_files_with_class("test", 13)

    print(f"Original train Bone_Loss files: {len(train_bone_loss_labels)}")
    print(f"Original test Bone_Loss files: {len(test_bone_loss_labels)}")

    target_test_count = 150
    if len(test_bone_loss_labels) < target_test_count:
        num_to_move = target_test_count - len(test_bone_loss_labels)
        random.seed(42)
        to_move = random.sample(train_bone_loss_labels, num_to_move)

        test_img_dir = data_root / "test" / "images"
        test_label_dir = data_root / "test" / "labels"
        test_img_dir.mkdir(parents=True, exist_ok=True)
        test_label_dir.mkdir(parents=True, exist_ok=True)

        for lf in to_move:
            shutil.move(str(lf), test_label_dir / lf.name)
            img_dir = data_root / "train" / "images"
            for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
                candidate = img_dir / (lf.stem + ext)
                if candidate.exists():
                    shutil.move(str(candidate), test_img_dir / candidate.name)
                    break
        print(f"Moved {len(to_move)} Bone_Loss images and labels from train to test.")
    else:
        print("Test set already contains sufficient Bone_Loss files.")

def convert_split(split_name):
    img_dir = data_root / split_name / "images"
    label_dir = data_root / split_name / "labels"

    copied = 0
    skipped = 0

    for label_file in sorted(label_dir.glob("*.txt")):
        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
            candidate = img_dir / (label_file.stem + ext)
            if candidate.exists():
                img_path = candidate
                break

        if img_path is None:
            skipped += 1
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue

        h, w, _ = img.shape
        idx = 0

        with open(label_file, "r") as handle:
            for line in handle:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                if cls_id not in CLASS_NAMES:
                    continue

                x_c, y_c, bw, bh = map(float, parts[1:5])
                x_c_px = x_c * w
                y_c_px = y_c * h
                w_px = bw * w
                h_px = bh * h

                pad_w = w_px * 0.15
                pad_h = h_px * 0.15

                xmin = int(x_c_px - (w_px / 2) - pad_w)
                xmax = int(x_c_px + (w_px / 2) + pad_w)
                ymin = int(y_c_px - (h_px / 2) - pad_h)
                ymax = int(y_c_px + (h_px / 2) + pad_h)

                xmin = max(0, xmin)
                ymin = max(0, ymin)
                xmax = min(w, xmax)
                ymax = min(h, ymax)

                if xmax <= xmin or ymax <= ymin:
                    continue

                crop = img[ymin:ymax, xmin:xmax]
                dest_folder = OUTPUT / split_name / CLASS_NAMES[cls_id]
                dest_folder.mkdir(parents=True, exist_ok=True)
                dest = dest_folder / f"{img_path.stem}_crop_{idx}.jpg"

                cv2.imwrite(str(dest), crop)
                copied += 1
                idx += 1

    print(f"{split_name:6s} -> {copied} crops made, {skipped} skipped")

print("Idempotent check and moving Bone_Loss from train to test in progress...")
move_bone_loss_to_test()

for split in ["train", "valid", "test"]:
    convert_split(split)

random.seed(42)
target_train_count = 2000
for cls_name in CLASS_NAMES.values():
    train_folder = OUTPUT / "train" / cls_name
    if train_folder.exists():
        crops = list(train_folder.glob("*.jpg"))
        if len(crops) > target_train_count:
            to_remove = random.sample(crops, len(crops) - target_train_count)
            for crop in to_remove:
                crop.unlink()
            print(f"Downsampled {cls_name} training set to {target_train_count} samples (removed {len(to_remove)}).")
        else:
            print(f"{cls_name} training set has {len(crops)} samples, no downsampling needed.")

train  → 14035 copies made, 0 skipped
valid  → 3970 copies made, 0 skipped
test   → 1994 copies made, 0 skipped

Done!


In [27]:
print(f"{'Disease':<22} {'train':>6} {'valid':>6} {'test':>6}")
print("-" * 44)

for cls_name in CLASS_NAMES.values():
    counts = []
    for split in ["train", "valid", "test"]:
        folder = OUTPUT / split / cls_name
        if folder.exists():
            counts.append(len(list(folder.glob("*.*"))))
        else:
            counts.append(0)
    print(f"{cls_name:<22} {counts[0]:>6} {counts[1]:>6} {counts[2]:>6}")

Disease                 train  valid   test
--------------------------------------------
Caries                   2182    614    269
Missing_Teeth            1230    257    173
Periapical_Lesion        1691    471    212
Impacted_Tooth           7655   2416   1340
Bone_Loss                1277    212      0
